In [1]:
import pandas as pd
from geopy.distance import geodesic
from scipy.spatial import cKDTree
import numpy as np


In [2]:
df = pd.read_csv('export_2025314.csv')

In [3]:
df

,OBJECTID,GF_SEQ_ID,BATCH_ID,ACTION,LAMP_1_ACTION,LAMP_2_ACTION,LAMP_3_ACTION,ASSETNUM,POLE_TYPE,POLE_NUMBER,...,CALC_DATA_INTEGRATION_DT,GLOBALID,CreationDate,Creator,EditDate,Editor,geometryType,x,y,crs
0,151681,22418,1279,none,none,none,none,P05P0006,TP-StdSteel,334,...,NaN,4bd54ee1-aa60-4e2b-8148-a1a5991b1464,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,5.990569e+06,2.093444e+06,102643
1,151682,22419,1279,none,none,none,none,P05P0007,TP-StdSteel,335,...,NaN,f7f88aec-b2b6-44a1-ae91-6abf8f24b24b,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,5.990562e+06,2.093369e+06,102643
2,151683,22420,1279,none,none,none,none,P05P0008,TP-StdSteel,336,...,NaN,d7c40434-6eb0-487a-9fbb-61cf6508d961,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,5.990557e+06,2.093287e+06,102643
3,151684,22421,1279,none,none,none,none,P05P0009,TP-StdSteel,337,...,NaN,fd0ddbff-1281-4719-8e39-1f78e338db5e,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,5.990557e+06,2.093226e+06,102643
4,151685,22422,1279,none,none,none,none,P05P0010,TP-StdSteel,341,...,NaN,0fe942d0-9658-4bb9-b5f3-452db8e48f23,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,5.990547e+06,2.093117e+06,102643
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25275,176956,9712,1279,none,none,none,none,G10P0123,SL-OctaConc,16,...,NaN,bcd81191-e017-4b4e-b763-29c89fc74ba8,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,6.006948e+06,2.111595e+06,102643
25276,176957,9713,1279,none,none,none,none,G10P0124,SL-StdSteel,21,...,NaN,0e356a86-26df-4d93-9668-aa24a9c5cbe5,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,6.006665e+06,2.111473e+06,102643
25277,176958,9714,1279,none,none,none,none,G10P0125,SL-RndConc,22,...,NaN,4e879f60-57f5-4c6d-b486-cc2192be9f2a,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,6.006791e+06,2.111520e+06,102643
25278,176959,9715,1279,none,none,none,none,G10P0126,SL-StdSteel,23,...,NaN,440ac7af-e64a-487d-85e3-25c223978cef,1649814105332,shreepad_sfpuc,1649814105332,shreepad_sfpuc,esriGeometryPoint,6.006412e+06,2.111438e+06,102643


In [4]:
df2 = pd.read_csv('globalid_latlon_mapping.csv')

In [5]:
df2


,GLOBALID,Longitude,Latitude
0,4bd54ee1-aa60-4e2b-8148-a1a5991b1464,53.814194,18.476888
1,f7f88aec-b2b6-44a1-ae91-6abf8f24b24b,53.814132,18.476251
2,d7c40434-6eb0-487a-9fbb-61cf6508d961,53.814088,18.475552
3,fd0ddbff-1281-4719-8e39-1f78e338db5e,53.814088,18.475035
4,0fe942d0-9658-4bb9-b5f3-452db8e48f23,53.813999,18.474107
...,...,...,...
25275,bcd81191-e017-4b4e-b763-29c89fc74ba8,53.961333,18.631470
25276,0e356a86-26df-4d93-9668-aa24a9c5cbe5,53.958787,18.630433
25277,4e879f60-57f5-4c6d-b486-cc2192be9f2a,53.959923,18.630832
25278,440ac7af-e64a-487d-85e3-25c223978cef,53.956520,18.630134


In [6]:
# Merge on GLOBALID
df_merged = df.merge(df2, on="GLOBALID", how="left")

In [7]:
df_merged.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25280 entries, 0 to 25279
Data columns (total 63 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   OBJECTID                     25280 non-null  int64  
 1   GF_SEQ_ID                    25280 non-null  int64  
 2   BATCH_ID                     25280 non-null  int64  
 3   ACTION                       25280 non-null  object 
 4   LAMP_1_ACTION                25280 non-null  object 
 5   LAMP_2_ACTION                25280 non-null  object 
 6   LAMP_3_ACTION                25280 non-null  object 
 7   ASSETNUM                     25280 non-null  object 
 8   POLE_TYPE                    25280 non-null  object 
 9   POLE_NUMBER                  25280 non-null  object 
 10  POLE_OWNER                   0 non-null      float64
 11  ARM_LENGTH                   13 non-null     object 
 12  MATERIAL                     25280 non-null  object 
 13  MATERIAL_REF_ID 

In [8]:
# Save as a new CSV
df_merged.to_csv("merged_dataset_with_latlon.csv", index=False)

In [9]:
df3 = pd.read_csv('stopidsforgoodschoolbus.csv')

In [19]:
df3.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1037 entries, 0 to 1036
Data columns (total 26 columns):
 #   Column                        Non-Null Count  Dtype  
---  ------                        --------------  -----  
 0   OBJECTID                      1037 non-null   int64  
 1   STOPNAME                      1037 non-null   object 
 2   TRAPEZESTOPABBR               1037 non-null   object 
 3   RUCUSSTOPABBR                 938 non-null    object 
 4   STOPID                        1037 non-null   int64  
 5   LATITUDE                      1037 non-null   float64
 6   LONGITUDE                     1037 non-null   float64
 7   ACCESSIBILITYMASK             182 non-null    float64
 8   ATSTREET                      1020 non-null   object 
 9   ONSTREET                      1032 non-null   object 
 10  POSITION                      938 non-null    object 
 11  ORIENTATION                   950 non-null    object 
 12  SERVICEPLANNINGSTOPTYPE       515 non-null    object 
 13  SHE

In [11]:
# Convert lat/lon to NumPy arrays
stops = np.array(df3[['LATITUDE', 'LONGITUDE']])
lights = np.array(df_merged[['Latitude', 'Longitude']])

In [13]:
# Build KDTree for fast nearest neighbor search
tree = cKDTree(lights)

In [15]:
# Query all stop points to find closest match
distances, indices = tree.query(stops, k=1, distance_upper_bound=0.0000057)  # ~2 feet in degrees


In [20]:
# Filter valid matches
valid_matches = distances < 0.0000057  
matched_stops = df3.iloc[valid_matches]
matched_merged = df_merged.iloc[indices[valid_matches]]

In [21]:
# Create result DataFrame
close_stops_df = pd.DataFrame({
    'STOPID': matched_stops['STOPID'].values,
    'STOPNAME': matched_stops['STOPNAME'].values,
    'Stop Latitude': matched_stops['LATITUDE'].values,
    'Stop Longitude': matched_stops['LONGITUDE'].values,
    'Matched Latitude': matched_merged['Latitude'].values,
    'Matched Longitude': matched_merged['Longitude'].values,
    'Distance (feet)': distances[valid_matches] * 364000  # Convert degrees to feet
})

In [22]:
# Save to CSV
close_stops_df.to_csv("stops_within_2_feet.csv", index=False)